# Experiments

This notebook runs the evaluation and ablation experiments for the Vasuki RL project: round-robin tournaments between agent types, learning-curve comparisons, a reward-shaping ablation, and a scalability (>8 grid) stretch experiment.

**Prerequisite:** several cells below load trained models from `models/dqn_random.zip` and `models/dqn_selfplay.zip`. These files are **not** produced automatically — you must first run the training scripts described in the project `README.md`:

```bash
python -m training.train_tabular
python -m training.train_dqn
python -m training.train_selfplay
```

Cells that depend on these artifacts are guarded with existence checks so the notebook does not crash if they are missing; they will simply skip with a printed message until you have trained the models.

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from stable_baselines3 import DQN

from vasuki.opponents import RandomOpponent, TabularOpponent, PolicySnapshotOpponent
from analysis.evaluate import round_robin, play_match

CONFIG = {"n": 8, "rewards": {"Food": 4, "Movement": -1, "Illegal": -2}, "game_length": 100}

MODELS_DIR = Path("models")
QTABLE_PATH = MODELS_DIR / "best_qtable.pkl"
DQN_RANDOM_PATH = MODELS_DIR / "dqn_random.zip"
DQN_SELFPLAY_PATH = MODELS_DIR / "dqn_selfplay.zip"

## 1. Load the four agent types

- `RandomOpponent` — always available, no artifacts required.
- `TabularOpponent` — needs `models/best_qtable.pkl` (produced by `training/train_tabular.py`).
- `PolicySnapshotOpponent` wrapping the DQN-vs-random model — needs `models/dqn_random.zip`.
- `PolicySnapshotOpponent` wrapping the self-play DQN model — needs `models/dqn_selfplay.zip`.

Any agent whose artifact is missing is skipped (with a printed warning) so the rest of the notebook still runs with whatever agents are available.

In [ ]:
agents = {"random": RandomOpponent()}

if QTABLE_PATH.exists():
    with open(QTABLE_PATH, "rb") as f:
        q_table = pickle.load(f)
    agents["tabular"] = TabularOpponent(q_table)
else:
    print(f"Skipping tabular agent: {QTABLE_PATH} not found")

if DQN_RANDOM_PATH.exists():
    agents["dqn_random"] = PolicySnapshotOpponent(DQN.load(str(DQN_RANDOM_PATH)))
else:
    print(f"Skipping dqn_random agent: {DQN_RANDOM_PATH} not found (run training/train_dqn.py)")

if DQN_SELFPLAY_PATH.exists():
    agents["dqn_selfplay"] = PolicySnapshotOpponent(DQN.load(str(DQN_SELFPLAY_PATH)))
else:
    print(f"Skipping dqn_selfplay agent: {DQN_SELFPLAY_PATH} not found (run training/train_selfplay.py)")

print("Loaded agents:", list(agents.keys()))

## 2. Round-robin tournament and win-rate heatmap

`round_robin(CONFIG, agents, n_games=100)` plays every ordered pair of distinct agents `n_games` times and returns a dict keyed by `(name_a, name_b)` mapping to A's win rate. We reshape that into a matrix and plot it as a heatmap.

In [ ]:
N_GAMES = 100
names = list(agents.keys())

if len(names) < 2:
    print("Need at least 2 agents to run a round robin; train more models first.")
    win_matrix = None
else:
    results = round_robin(CONFIG, agents, n_games=N_GAMES)

    win_matrix = np.full((len(names), len(names)), np.nan)
    for i, a in enumerate(names):
        for j, b in enumerate(names):
            if a == b:
                continue
            win_matrix[i, j] = results[(a, b)]

    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(win_matrix, cmap="viridis", vmin=0, vmax=1)
    ax.set_xticks(range(len(names)))
    ax.set_yticks(range(len(names)))
    ax.set_xticklabels(names, rotation=45, ha="right")
    ax.set_yticklabels(names)
    ax.set_xlabel("Opponent (B)")
    ax.set_ylabel("Agent (A)")
    ax.set_title(f"Win rate of A vs B ({N_GAMES} games/pair)")

    for i in range(len(names)):
        for j in range(len(names)):
            val = win_matrix[i, j]
            if not np.isnan(val):
                ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                        color="white" if val < 0.5 else "black")

    fig.colorbar(im, ax=ax, label="Win rate (A)")
    fig.tight_layout()
    plt.show()

## 3. Learning curves: Tabular vs DQN

Neither `training/train_tabular.py` nor `training/train_dqn.py` / `training/train_selfplay.py` currently persist per-episode reward history to disk — `train_tabular` only saves the final Q-table (`models/best_qtable.pkl`), and the DQN scripts only save the final policy (`models/dqn_random.zip`, `models/dqn_selfplay.zip`) via Stable-Baselines3's `model.save`.

To plot a learning curve you would need to instrument training to record rewards, e.g.:

- In `train_tabular`, accumulate the per-episode total reward for agent A into a list and `np.save("models/tabular_rewards.npy", np.array(episode_rewards))` after the training loop.
- For the DQN scripts, wrap the environment with SB3's `Monitor` (from `stable_baselines3.common.monitor`) which writes a `monitor.csv` with per-episode reward/length, or pass a custom callback that appends `episode` info to a list and save it as `models/dqn_random_rewards.npy` / `models/dqn_selfplay_rewards.npy`.

The cell below looks for these `.npy` files and plots whichever are present; it will not raise if none exist yet.

In [ ]:
curve_paths = {
    "tabular": MODELS_DIR / "tabular_rewards.npy",
    "dqn_random": MODELS_DIR / "dqn_random_rewards.npy",
    "dqn_selfplay": MODELS_DIR / "dqn_selfplay_rewards.npy",
}

fig, ax = plt.subplots(figsize=(7, 4))
found_any = False
for label, path in curve_paths.items():
    if path.exists():
        rewards = np.load(path)
        # Light smoothing via a rolling mean for readability
        window = max(1, len(rewards) // 100)
        smoothed = np.convolve(rewards, np.ones(window) / window, mode="valid")
        ax.plot(smoothed, label=label)
        found_any = True
    else:
        print(f"Skipping {label}: {path} not found (see markdown above for how to produce it)")

if found_any:
    ax.set_xlabel("Episode")
    ax.set_ylabel("Reward (smoothed)")
    ax.set_title("Learning curves: Tabular vs DQN")
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    print("No reward-history files found; nothing to plot yet.")

## 4. Reward-shaping ablation: with vs without distance-to-food shaping

The current `Vasuki` environment (`vasuki/env.py`) only exposes a fixed reward dict with three keys — `Food`, `Movement`, `Illegal` — via `self.rewards`. There is no built-in hook for continuous distance-to-food shaping in `Vasuki._reward_` or in `GymWrapper.step`; shaping would need to be added as a wrapper around `GymWrapper` (e.g. adding `-alpha * distance_to_nearest_food` to the reward each step).

The cell below defines a small optional shaping wrapper inline and trains two short DQN runs (a few thousand timesteps each) — one on the plain `GymWrapper`, one on the shaped version — purely to illustrate the ablation methodology. Because no shaping hook exists in the shipped environment, this is additive/experimental code local to the notebook, not a claim that the trained package supports it out of the box. The cell is wrapped in a try/except so a failure (e.g. missing optional deps, long runtime) degrades gracefully instead of crashing the notebook.

In [ ]:
import gymnasium as gym
from vasuki.gym_wrapper import GymWrapper
from vasuki.features import to_vector

SHAPING_TIMESTEPS = 3_000  # kept small so the ablation cell runs quickly


class DistanceShapedWrapper(gym.Wrapper):
    """Adds a small negative-distance-to-food shaping term on top of the base reward.

    This is notebook-local experimental code: vasuki/gym_wrapper.py does not define a shaping
    hook, so shaping is layered on here via gym.Wrapper rather than by modifying the package.
    """

    def __init__(self, env, alpha=0.05):
        super().__init__(env)
        self.alpha = alpha

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        food_dx, food_dy = obs[0], obs[1]
        distance = float(np.hypot(food_dx, food_dy))
        shaped_reward = reward - self.alpha * distance
        return obs, shaped_reward, terminated, truncated, info


try:
    from stable_baselines3 import DQN
    from vasuki.opponents import RandomOpponent

    def _make_model(env):
        return DQN("MlpPolicy", env, verbose=0, seed=0, learning_rate=1e-3,
                   buffer_size=10_000, learning_starts=200, batch_size=64, gamma=0.95)

    env_plain = GymWrapper(CONFIG, RandomOpponent())
    model_plain = _make_model(env_plain)
    model_plain.learn(total_timesteps=SHAPING_TIMESTEPS)

    env_shaped = DistanceShapedWrapper(GymWrapper(CONFIG, RandomOpponent()))
    model_shaped = _make_model(env_shaped)
    model_shaped.learn(total_timesteps=SHAPING_TIMESTEPS)

    from vasuki.opponents import PolicySnapshotOpponent
    ablation_agents = {
        "plain": PolicySnapshotOpponent(model_plain),
        "shaped": PolicySnapshotOpponent(model_shaped),
        "random": RandomOpponent(),
    }
    ablation_results = round_robin(CONFIG, ablation_agents, n_games=20)
    print("Reward-shaping ablation (win rate vs random, 20 games):")
    print("  plain :", ablation_results[("plain", "random")])
    print("  shaped:", ablation_results[("shaped", "random")])
except Exception as exc:  # pragma: no cover - notebook convenience guard
    print(f"Reward-shaping ablation skipped due to error: {exc}")

## 5. Stretch: scaling beyond an 8x8 grid

`Vasuki` accepts an arbitrary grid size `n` via its constructor. Below we build a 12x12 environment (`n=12`) and train a short DQN run on it to sanity-check that the DQN pipeline scales to larger grids without code changes.

**Why not tabular Q-learning at this scale?** `training/train_tabular.py`'s state representation (`vasuki.features.get_input_states`) is `(food_coord, enemy_coord, head)`, where `food_coord` and `enemy_coord` are each pairs of relative coordinates ranging over `-(n-1) .. (n-1)`, i.e. `2n - 1` values per axis, and `head` has 4 values. The exhaustive table built by `_init_qtable` therefore has `(2n-1)^4 * 4` entries. For `n=8` that is already `15^4 * 4 = 202{,}500` entries; for `n=12` it becomes `23^4 * 4 = 1{,}119{,}364` entries, and it keeps growing as `O(n^4)`. DQN instead uses a fixed-size 8-dimensional feature vector (`vasuki.features.to_vector`) and a small MLP with a constant parameter count regardless of `n`, so it scales to larger grids without the combinatorial blow-up that a full tabular Q-table would need.

In [ ]:
LARGE_CONFIG = {"n": 12, "rewards": {"Food": 4, "Movement": -1, "Illegal": -2}, "game_length": 100}
STRETCH_TIMESTEPS = 3_000  # short illustrative run, not a full training budget

try:
    from stable_baselines3 import DQN
    from vasuki.opponents import RandomOpponent

    large_env = GymWrapper(LARGE_CONFIG, RandomOpponent())
    large_model = DQN("MlpPolicy", large_env, verbose=0, seed=0, learning_rate=1e-3,
                      buffer_size=10_000, learning_starts=200, batch_size=64, gamma=0.95)
    large_model.learn(total_timesteps=STRETCH_TIMESTEPS)
    print("Trained a short DQN policy on a 12x12 grid successfully.")

    n = LARGE_CONFIG["n"]
    tabular_entries = (2 * n - 1) ** 4 * 4
    print(f"An exhaustive tabular Q-table for n={n} would need (2n-1)^4 * 4 = {tabular_entries:,} entries,")
    print("versus DQN's fixed-size MLP parameter count (independent of n).")
except Exception as exc:  # pragma: no cover - notebook convenience guard
    print(f"12x12 stretch experiment skipped due to error: {exc}")